# DesignBench — Google Colab Setup
**Benchmark for MLLM-based Front-end Code Generation**

https://github.com/WebPAI/DesignBench

---
Run cells **in order**, top to bottom.

## CELL 1 — Clone Repo & Patch Sources

Clones the upstream repo, then applies all patches needed for Colab:
- Lazy LLM imports (avoids needing unused SDKs)
- Qwen international API endpoint
- Chrome instead of Firefox for Selenium
- Fix hardcoded macOS node path in AST evaluator
- Fix evaluator data paths to match actual dataset structure
- Add missing `re_calculate` global

In [ ]:
import os, re

if not os.path.exists('/content/DesignBench'):
    !git clone https://github.com/WebPAI/DesignBench.git /content/DesignBench
else:
    print('Repo already cloned, skipping.')

%cd /content/DesignBench
!mkdir -p data code/evaluator/res code/evaluator/tmp tmp
print('Repo ready.')

# ── Patch 1: Lazy LLM imports ─────────────────────────────────────
with open('/content/DesignBench/code/mllm/__init__.py', 'w') as f:
    f.write('''from .base import MLLMChat

__all__ = [
    "MLLMChat", "OpenAIChat", "AnthropicChat", "MistralChat",
    "GeminiChat", "DeepInfraChat", "QwenChat",
]

def get_model(model_name: str, **kwargs) -> MLLMChat:
    match model_name:
        case "gpt-4o-2024-11-20":
            from .openai_chat import OpenAIChat
            return OpenAIChat(model_name, **kwargs)
        case "claude-3-7-sonnet-20250219":
            from .anthropic_chat import AnthropicChat
            return AnthropicChat(model_name, **kwargs)
        case "gemini-2.0-flash":
            from .gemini_chat import GeminiChat
            return GeminiChat(model_name, **kwargs)
        case "qwen2.5-vl-72b-instruct" | "qwen2.5-vl-7b-instruct":
            from .platform_api import QwenChat
            return QwenChat(model_name, **kwargs)
        case "meta-llama/Llama-3.2-90B-Vision-Instruct" | "meta-llama/Llama-3.2-11B-Vision-Instruct":
            from .platform_api import DeepInfraChat
            return DeepInfraChat(model_name, **kwargs)
        case "pixtral-large-latest" | "pixtral-12b-2409":
            from .mistral_chat import MistralChat
            return MistralChat(model_name, **kwargs)
        case _:
            raise ValueError(f"Unsupported model name: {model_name}")
''')
print('Patched: mllm/__init__.py (lazy imports)')

# ── Patch 2: Qwen international endpoint ─────────────────────────
api_path = '/content/DesignBench/code/mllm/platform_api.py'
with open(api_path, 'r') as f:
    content = f.read()
content = content.replace(
    'https://dashscope.aliyuncs.com/compatible-mode/v1',
    'https://dashscope-intl.aliyuncs.com/compatible-mode/v1'
)
with open(api_path, 'w') as f:
    f.write(content)
print('Patched: platform_api.py (Qwen intl endpoint)')

# ── Patch 3: Chrome instead of Firefox ────────────────────────────
mu_path = '/content/DesignBench/code/evaluator/metric_utils.py'
with open(mu_path, 'r') as f:
    content = f.read()
content = content.replace(
    'from selenium.webdriver.firefox.service import Service',
    'from selenium.webdriver.chrome.service import Service'
)
content = content.replace(
    'from selenium.webdriver.firefox.options import Options',
    'from selenium.webdriver.chrome.options import Options'
)
content = content.replace(
    'service = Service(executable_path=firefox_path)',
    'service = Service()'
)
content = content.replace(
    'driver = webdriver.Firefox(options=options, service=service)',
    'options.add_argument(\"--no-sandbox\")\n        options.add_argument(\"--disable-dev-shm-usage\")\n        driver = webdriver.Chrome(options=options, service=service)'
)
content = content.replace(
    "browser_name='firefox'",
    "browser_name='chrome'"
)
with open(mu_path, 'w') as f:
    f.write(content)
print('Patched: metric_utils.py (Chrome instead of Firefox)')

# ── Patch 4: Fix node path + tmp dir in AST evaluator ────────────
ast_path = '/content/DesignBench/code/evaluator/metric_ast.py'
with open(ast_path, 'r') as f:
    content = f.read()
content = content.replace(
    "/Users/whalexiao/.nvm/versions/node/v18.19.0/bin/node",
    "/root/.nvm/versions/node/v20.20.2/bin/node"
)
content = content.replace(
    "dir='./tmp'",
    "dir='/content/DesignBench/tmp'"
)
with open(ast_path, 'w') as f:
    f.write(content)
print('Patched: metric_ast.py (node path + tmp dir)')

# ── Patch 5: Fix DesignBench_Path + evaluator data paths ─────────
cfg_path = '/content/DesignBench/code/evaluator/config.py'
with open(cfg_path, 'r') as f:
    content = f.read()
content = re.sub(
    r'DesignBench_Path\s*=\s*["\'].*?["\']',
    'DesignBench_Path = "/content/DesignBench/"',
    content
)
content = content.replace('"data/DesignGeneration/"', '"data/generation/"')
content = content.replace('"data/DesignEdit/"', '"data/edit/"')
content = content.replace('"data/DesignRepair/"', '"data/repair/"')
with open(cfg_path, 'w') as f:
    f.write(content)
print('Patched: config.py (DesignBench_Path + data folder paths)')

# ── Patch 6: Add missing re_calculate global ─────────────────────
eval_path = '/content/DesignBench/code/evaluator/main.py'
with open(eval_path, 'r') as f:
    content = f.read()
if 're_calculate = ' not in content.split('if __name__')[0]:
    content = 're_calculate = False\n' + content
    with open(eval_path, 'w') as f:
        f.write(content)
print('Patched: evaluator/main.py (re_calculate global)')

print('\nAll patches applied.')

## CELL 2 — Install Chrome

In [ ]:
!apt-get update --fix-missing -qq
!apt-get install -f -qq
!wget -q https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!apt-get install -y ./google-chrome-stable_current_amd64.deb -qq
!rm google-chrome-stable_current_amd64.deb
!google-chrome --version

## CELL 3 — Install Node v20 + npm dependencies

In [ ]:
# ── nvm + Node v20 ────────────────────────────────────────────────
!curl -fsSL https://raw.githubusercontent.com/nvm-sh/nvm/v0.39.7/install.sh | bash -s -- --no-use
!bash -c 'source /root/.nvm/nvm.sh && nvm install 20 && nvm use 20 && node --version && npm --version'

# ── single-file-cli ───────────────────────────────────────────────
!bash -c 'source /root/.nvm/nvm.sh && nvm use 20 && npm install -g single-file-cli'

# ── Web framework dependencies ────────────────────────────────────
print('Installing React dependencies...')
!bash -c 'source /root/.nvm/nvm.sh && nvm use 20 && cd /content/DesignBench/web/my-react-app && npm install --silent'

print('Installing Vue dependencies...')
!bash -c 'source /root/.nvm/nvm.sh && nvm use 20 && cd /content/DesignBench/web/my-vue-app && npm install --silent'

print('Installing Angular dependencies...')
!bash -c 'source /root/.nvm/nvm.sh && nvm use 20 && npm install -g @angular/cli --silent && cd /content/DesignBench/web/my-angular-app && npm install --silent'

# ── Evaluator npm dependencies + AST parsers ──────────────────────
print('Installing evaluator dependencies...')
!bash -c 'source /root/.nvm/nvm.sh && nvm use 20 && cd /content/DesignBench/code/evaluator && npm install --silent'
!bash -c 'source /root/.nvm/nvm.sh && nvm use 20 && npm install @babel/parser @vue/compiler-dom parse5 --prefix /content/DesignBench'

print('Node setup complete.')

## CELL 4 — Install Python Packages

In [ ]:
!pip install -q \
  anthropic==0.68.0 \
  openai>=1.50,<2 \
  google-generativeai==0.8.5 \
  google-api-python-client==2.181.0 \
  google-auth==2.40.3 \
  selenium==4.35.0 \
  opencv-python==4.11.0.86 \
  scikit-image==0.25.2 \
  pillow==11.3.0 \
  numpy \
  scipy==1.15.3 \
  openai-clip==1.0.1 \
  ftfy==6.3.1 \
  regex==2025.9.1 \
  tqdm==4.67.1 \
  requests==2.32.5 \
  retry==0.9.2 \
  imageio==2.37.0 \
  pydantic==2.11.9 \
  httpx==0.28.1 \
  filelock==3.19.1 \
  fsspec==2025.9.0 \
  networkx==3.4.2 \
  sympy==1.14.0 \
  jinja2==3.1.6 \
  packaging==25.0 \
  websocket-client==1.8.0 \
  trio==0.30.0 \
  trio-websocket==0.12.2 \
  python-dotenv

!pip install -q torch torchvision

import torch, selenium, cv2, PIL, numpy
print(f'Python packages installed. torch: {torch.__version__}, numpy: {numpy.__version__}')

## CELL 5 — Mount Google Drive & Extract Dataset

Upload your dataset zip (e.g. `data.zip`) to Google Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os, zipfile

zip_src = '/content/drive/MyDrive/designbench_data.zip'
data_dst = '/content/DesignBench/'

if not os.path.exists(zip_src):
    print(f'Could not find {zip_src}')
    print('Check your Drive: !ls "/content/drive/MyDrive/"')
else:
    print('Extracting dataset...')
    with zipfile.ZipFile(zip_src, 'r') as z:
        z.extractall(data_dst)
    print('Done! Contents of data/:')
    !ls /content/DesignBench/data/

## CELL 6 — Set API Keys

Fill in the keys you need. Leave others as empty strings.

In [ ]:
import os, json

# ── Fill in your keys here ────────────────────────────────────────
OPENAI_API_KEY      = ""   # gpt-4o etc.
ANTHROPIC_API_KEY   = ""   # claude
GEMINI_API_KEY      = ""   # gemini
QWEN_API_KEY        = ""   # qwen
DEEPINFRA_API_KEY   = ""   # llama via deepinfra
MISTRAL_API_KEY     = ""   # mistral / pixtral

# ── Write .env file ───────────────────────────────────────────────
env_content = f"""OPENAI_API_KEY=\"{OPENAI_API_KEY}\"
ANTHROPIC_API_KEY=\"{ANTHROPIC_API_KEY}\"
GEMINI_API_KEY=\"{GEMINI_API_KEY}\"
QWEN_API_KEY=\"{QWEN_API_KEY}\"
DEEPINFRA_API_KEY=\"{DEEPINFRA_API_KEY}\"
MISTRAL_API_KEY=\"{MISTRAL_API_KEY}\"
"""
with open('/content/DesignBench/.env', 'w') as f:
    f.write(env_content)

# ── Write code/prompting/key.json ─────────────────────────────────
key_json = {
    "gpt":     OPENAI_API_KEY,
    "claude":  ANTHROPIC_API_KEY,
    "gemini":  GEMINI_API_KEY,
    "qwen":    QWEN_API_KEY,
    "llama":   DEEPINFRA_API_KEY,
    "mistral": MISTRAL_API_KEY,
}
os.makedirs('/content/DesignBench/code/prompting', exist_ok=True)
with open('/content/DesignBench/code/prompting/key.json', 'w') as f:
    json.dump({k: v for k, v in key_json.items() if v}, f, indent=2)

# ── Also set as env vars for this session ─────────────────────────
os.environ.update({
    'OPENAI_API_KEY':    OPENAI_API_KEY,
    'ANTHROPIC_API_KEY': ANTHROPIC_API_KEY,
    'GEMINI_API_KEY':    GEMINI_API_KEY,
    'QWEN_API_KEY':      QWEN_API_KEY,
    'DEEPINFRA_API_KEY': DEEPINFRA_API_KEY,
    'MISTRAL_API_KEY':   MISTRAL_API_KEY,
})

filled = [k for k, v in key_json.items() if v]
empty  = [k for k, v in key_json.items() if not v]
print(f'Keys set:   {filled if filled else "none"}')
print(f'Keys empty: {empty if empty else "none"}')

## CELL 7 — Sanity Check (optional)

In [ ]:
import subprocess, os

def check(name, cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    out = (r.stdout or r.stderr).strip().split('\n')[0]
    icon = 'OK' if r.returncode == 0 else 'FAIL'
    print(f'[{icon}] {name:20s} {out}')

print('-- Browsers --')
check('Chrome',        'google-chrome --version')

print('\n-- Node & npm --')
check('Node',          "bash -c 'source /root/.nvm/nvm.sh && node --version'")
check('npm',           "bash -c 'source /root/.nvm/nvm.sh && npm --version'")

print('\n-- Python packages --')
for pkg in ['torch', 'selenium', 'cv2', 'PIL', 'numpy', 'anthropic', 'openai']:
    try:
        mod = __import__(pkg)
        ver = getattr(mod, '__version__', 'ok')
        print(f'[OK]   {pkg:20s} {ver}')
    except ImportError:
        print(f'[FAIL] {pkg:20s} NOT FOUND')

print('\n-- Paths --')
for p in ['/content/DesignBench/data', '/content/DesignBench/.env',
          '/content/DesignBench/code/prompting/key.json']:
    exists = os.path.exists(p)
    icon = 'OK' if exists else 'FAIL'
    print(f'[{icon}] {p}')

---
## CELL 8 — Run LLM Task

Uncomment ONE of the run configurations below.
- **Option A**: Quick test — 2 samples per framework (repair only)
- **Option B**: Single framework, single task, custom range
- **Option C**: All frameworks, all tasks, full dataset

In [ ]:
import sys
sys.path.insert(0, '/content/DesignBench/code')

from runner.main import Runner
from utils import Framework, Task, Mode

MODEL = "qwen2.5-vl-72b-instruct"  # see mllm/__init__.py for all options

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Option A: Quick test — 2 samples from each framework (repair only)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
for fw in [Framework.REACT, Framework.VUE, Framework.ANGULAR, Framework.VANILLA]:
    runner = Runner(MODEL, framework=fw, stream=True, print_content=True)
    runner.run(task=Task.REPAIR, output_framework=fw, mode=Mode.BOTH, max_workers=2, execution_range=(1, 3))

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Option B: Single framework, single task, custom range
#   Uncomment below and comment out Option A
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# runner = Runner(MODEL, framework=Framework.REACT, stream=True, print_content=True)
# runner.run(task=Task.REPAIR, output_framework=Framework.REACT, mode=Mode.BOTH, max_workers=5, execution_range=(1, 6))

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Option C: All frameworks, all tasks, full dataset
#   Uncomment below and comment out Option A
#   WARNING: this calls the LLM hundreds of times
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# for fw in [Framework.REACT, Framework.VUE, Framework.ANGULAR, Framework.VANILLA]:
#     runner = Runner(MODEL, framework=fw, stream=True, print_content=True)
#     # Generation can output to any framework; edit/repair must match
#     runner.run(task=Task.GENERATION, output_framework=fw, mode=Mode.IMAGE, max_workers=5)
#     runner.run(task=Task.EDIT, output_framework=fw, mode=Mode.BOTH, max_workers=5)
#     runner.run(task=Task.REPAIR, output_framework=fw, mode=Mode.BOTH, max_workers=5)

---
## CELL 9 — Start Dev Servers

Required before evaluation (for rendering React/Vue/Angular code).
Re-run this cell after any kernel restart. Vanilla doesn't need a dev server.

In [ ]:
import subprocess, time

# Start all three framework dev servers in background
for app in ["my-react-app", "my-vue-app"]:
    subprocess.Popen(
        f'bash -c "source /root/.nvm/nvm.sh && nvm use 20 && cd /content/DesignBench/web/{app} && npm run dev"',
        shell=True,
        stdout=open(f'/tmp/{app}.log', 'w'),
        stderr=subprocess.STDOUT
    )

subprocess.Popen(
    'bash -c "source /root/.nvm/nvm.sh && nvm use 20 && cd /content/DesignBench/web/my-angular-app && ng serve"',
    shell=True,
    stdout=open('/tmp/angular.log', 'w'),
    stderr=subprocess.STDOUT
)

time.sleep(20)

# Verify all servers are up
for name, port in [("React", 3000), ("Vue", 5173), ("Angular", 4200)]:
    r = subprocess.run(f"curl -s -o /dev/null -w '%{{http_code}}' http://localhost:{port}", shell=True, capture_output=True, text=True)
    status = r.stdout.strip()
    icon = "OK" if status == "200" else "FAIL"
    print(f"[{icon}] {name:10s} http://localhost:{port}  (status {status})")

## CELL 10 — Evaluate

Symlinks runner output into the path the evaluator expects, then runs evaluation.
- **Option A** matches Cell 8 Option A: 2 samples from each framework (repair)
- **Option B**: Evaluate a single sample
- **Option C**: Evaluate all samples for all frameworks

In [ ]:
import sys, os
sys.path.insert(0, '/content/DesignBench/code')
sys.path.insert(0, '/content/DesignBench/code/evaluator')

# Symlink runner results into evaluator expected paths
for task_dir in ['repair', 'generation', 'edit']:
    src = f'/content/DesignBench/results/{task_dir}'
    dst = f'/content/DesignBench/data/{task_dir}/RepairResults' if task_dir == 'repair' else f'/content/DesignBench/data/{task_dir}/Results'
    if os.path.exists(src):
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        os.system(f'mkdir -p {dst} && ln -sf {src}/* {dst}/')

from evaluator.main import get_repair_metric

MODEL = "qwen2.5-vl-72b-instruct"

# Sample counts per framework for repair task
REPAIR_RANGES = {"react": 28, "vue": 27, "angular": 28, "vanilla": 28}

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Option A: Quick test — 2 samples from each framework (matches Cell 8 Option A)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
for fw in REPAIR_RANGES:
    print(f"\n=== {fw} ===")
    for i in range(1, 3):
        try:
            metric = get_repair_metric(web_name=str(i), model_name=MODEL, framework=fw, mode="both", llm_judge_flag=False)
            print(f"  Sample {i}: CLIP={metric.get('clip_similarity', 'N/A'):.3f}  SSIM={metric.get('structure_similarity', 'N/A'):.3f}  AST-ES={metric.get('ast_code_content_weighted_score', 'N/A')}")
        except Exception as e:
            print(f"  Sample {i}: SKIP ({e})")

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Option B: Evaluate a single sample
#   Uncomment below and comment out Option A
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# metric = get_repair_metric(web_name="1", model_name=MODEL, framework="react", mode="both", llm_judge_flag=False)
# print(metric)

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Option C: All samples, all frameworks
#   Uncomment below and comment out Option A
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# for fw, count in REPAIR_RANGES.items():
#     print(f"\n=== {fw} ===")
#     for i in range(1, count + 1):
#         try:
#             metric = get_repair_metric(web_name=str(i), model_name=MODEL, framework=fw, mode="both", llm_judge_flag=False)
#             print(f"  Sample {i}: CLIP={metric.get('clip_similarity', 'N/A'):.3f}  SSIM={metric.get('structure_similarity', 'N/A'):.3f}  AST-ES={metric.get('ast_code_content_weighted_score', 'N/A')}")
#         except Exception as e:
#             print(f"  Sample {i}: SKIP ({e})")